# Train a Model on MNIST with FiftyOne and Torch

## Import Libraries

In [1]:
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.utils.random as four

/home/spjor/PycharmProjects/dat255-pose-estimation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import urllib.request

To run this recipe, you’ll need the mnist_training.py script, which contains a simple PyTorch training loop. The following cell will automatically download the file into your working directory so it can be imported directly.

In [3]:
url = "https://cdn.voxel51.com/tutorials_torch_dataset_examples/notebook_simple_training_example/mnist_training.py"
urllib.request.urlretrieve(url, "mnist_training.py")

('mnist_training.py', <http.client.HTTPMessage at 0x79d588038c80>)

In [4]:
url = "https://cdn.voxel51.com/tutorials_torch_dataset_examples/notebook_the_cache_field_names_argument/utils.py"
urllib.request.urlretrieve(url, "utils.py")

('utils.py', <http.client.HTTPMessage at 0x79d5880fe870>)

In [5]:

from notebooks import mnist_training

In [6]:
torch.multiprocessing.set_start_method('forkserver') #  Doesnt work on windows
torch.multiprocessing.set_forkserver_preload(['torch', 'fiftyone'])

## Basic Training Example on MNIST

In [7]:
mnist = foz.load_zoo_dataset("mnist")
mnist.persistent = True

Split 'train' already downloaded
Split 'test' already downloaded
Loading existing dataset 'mnist'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


In [8]:
session = fo.launch_app(mnist, auto=False)

Session launched. Run `session.show()` to open the App in a cell output.


Now let’s say that for our training, we want to define some random subset of our trainset to be a validation set. We can easily do this with FiftyOne.

In [9]:
# remove existing 'train' or 'validation' tags if they exist
mnist.untag_samples(['train', 'validation'])

# create a random split, just on the samples not tagged 'test'
not_test = mnist.match_tags('test', bool=False)
four.random_split(not_test, {'train' : 0.9, 'validation' : 0.1})
print(mnist.count_sample_tags())

{'validation': 6000, 'train': 54000, 'test': 10000}


In [10]:
# subset if we want it
samples = []
samples += mnist.match_tags('train').take(1000).values('id')
for tag in ['test', 'validation']:
    samples += mnist.match_tags(tag).values('id')

subset = mnist.select(samples)

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [12]:
import os
current_path = os.getcwd()

path_to_save_weights = current_path + "/mnist_weights"

In [13]:
mnist_training.main(subset, 10, 10, device, path_to_save_weights)

Average Validation Loss =   2.200046: 100%|██████████| 375/375 [00:01<00:00, 252.47it/s]


New best lost achieved : 2.125579357147217. Saving model...


Average Validation Loss =   0.398228: 100%|██████████| 375/375 [00:01<00:00, 260.69it/s]


New best lost achieved : 0.40538060665130615. Saving model...


Average Validation Loss =   0.239873: 100%|██████████| 375/375 [00:01<00:00, 241.38it/s]


...

New best lost achieved : 0.20123420655727386. Saving model...


Average Validation Loss =   0.134998: 100%|██████████| 375/375 [00:01<00:00, 231.01it/s]


New best lost achieved : 0.13465452194213867. Saving model...


Average Validation Loss =   0.151300: 100%|██████████| 625/625 [00:06<00:00, 90.16it/s]


Final Test Results:
Loss = 0.15188241004943848
              precision    recall  f1-score   support

    0 - zero       0.96      0.99      0.97       980
     1 - one       0.99      0.99      0.99      1135
     2 - two       0.95      0.94      0.95      1032
   3 - three       0.97      0.89      0.93      1010
    4 - four       0.94      0.99      0.96       982
    5 - five       0.91      0.98      0.94       892
     6 - six       0.99      0.96      0.97       958
   7 - seven       0.98      0.93      0.95      1028
   8 - eight       0.90      0.97      0.93       974
    9 - nine       0.96      0.91      0.94      1009

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.96      0.95      0.95     10000

